# 98. Macro & Market Regime 데이터 구축

## 📋 개요
종목별 데이터 수집에 앞서, 한국 시장의 전역적(Global) 환경 변수인 거시 경제 지표와 Market Regime(강세/약세/보합장) 시그널을 계산하여 별도로 저장합니다.

## ✨ 핵심 파이프라인 및 업데이트
- **3단계 Fallback 수집 (v3.8.1)**: `FinanceDataReader` API 호출 실패 시 `data/99_meta/*.csv`의 로컬 백업 데이터를 자동으로 파싱하여 파이프라인 중단을 방지합니다.
- **KOSPI 거래일 동기화**: 모든 해외 지표(S&P500, VIX, 환율 등)는 KOSPI 개장일을 기준으로 재인덱싱되며, KOSPI 영업일인데 해외 지표가 없는 날(해외 휴장)은 직전 거래일 값으로 `ffill` 처리됩니다.
- **Regime 판단**: KOSPI 200일 이동평균선과 장단기 변동성을 종합하여 Bull(1), Bear(-1), Neutral(0) 상태를 산출합니다.

In [ ]:
import re
import pandas as pd
import numpy as np
import FinanceDataReader as fdr
from datetime import datetime, timedelta
from pathlib import Path

# 설정 로드 및 경로 준비 (기존 config 활용)
from src.utils.config import load_config, ProjectPaths
cfg = load_config()
paths = ProjectPaths.from_config(cfg)

# 매크로 데이터를 저장할 폴더
meta_dir = Path(cfg['paths'].get('meta_dir', 'data/99_meta'))
macro_filepath = meta_dir / "macro_regime.parquet"

print(f"📁 매크로 데이터 저장 경로: {macro_filepath}")

## 1️⃣ 매크로 지표 수집 및 Fallback 헬퍼 정의
네트워크 장애 시 CSV를 읽어오는 헬퍼 함수를 정의하고, 4대 핵심 지표(KOSPI, S&P500, USD/KRW, VIX)를 수집합니다.
- CSV 파싱 시 정규식을 통해 날짜(`YYYY. M. D`)만 추출하고 시간은 무시합니다.
- 동일 날짜에 여러 데이터가 존재하면 가장 마지막(장 마감) 데이터를 사용합니다.

In [ ]:
def _parse_csv_date(date_str: str) -> pd.Timestamp | None:
    m = re.match(r'(\d{4})\s*\.\s*(\d{1,2})\s*\.\s*(\d{1,2})', str(date_str))
    if m:
        return pd.Timestamp(f"{m.group(1)}-{m.group(2).zfill(2)}-{m.group(3).zfill(2)}")
    return None

def load_fallback_csv(indicator: str, meta_dir: Path) -> pd.Series:
    csv_path = meta_dir / f"{indicator}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Fallback CSV 없음: {csv_path}")

    df = pd.read_csv(csv_path)
    df['Date'] = df['Date'].apply(_parse_csv_date)
    df = df.dropna(subset=['Date'])
    df = df.sort_values('Date').groupby('Date', sort=True).last()
    
    series = df['Close'].astype(float)
    series.index = pd.to_datetime(series.index)
    series.index.name = 'Date'
    series.name = indicator
    return series

base_start = pd.to_datetime(cfg['data_collection']['start_date'])
base_end   = pd.to_datetime(cfg['data_collection']['end_date'])
fetch_start = (base_start - timedelta(days=364)).strftime('%Y-%m-%d')
fetch_end   = (base_end + timedelta(days=1)).strftime('%Y-%m-%d')

print(f"📥 데이터 수집 중... ({fetch_start} ~ {fetch_end})")

macro_raw = {}
sources   = {}

# KOSPI 수집 (기준 인덱스 설정)
try:
    macro_raw['kospi'] = fdr.DataReader('KS11', fetch_start, fetch_end)['Close']
    if macro_raw['kospi'].empty: raise ValueError("빈 데이터")
    sources['kospi'] = 'API'
except Exception as e:
    print(f"  ⚠️ KOSPI API 실패 ({e}) → CSV fallback")
    macro_raw['kospi'] = load_fallback_csv('kospi', meta_dir)
    sources['kospi'] = 'CSV'

kospi_series = macro_raw['kospi'].copy()
kospi_series.index = pd.to_datetime(kospi_series.index)
kospi_series = kospi_series[(kospi_series.index >= fetch_start) & (kospi_series.index <= fetch_end)]
kospi_index = kospi_series.index

# 해외 지표 수집 루프
symbols = {'sp500': 'US500', 'usd_krw': 'USD/KRW', 'vix': 'FRED:VIXCLS'}
for key, symbol in symbols.items():
    try:
        series = fdr.DataReader(symbol, fetch_start, fetch_end)
        col_name = 'VIXCLS' if key == 'vix' else 'Close'
        if series[col_name].empty: raise ValueError("빈 데이터")
        macro_raw[key] = series[col_name]
        sources[key] = 'API'
    except Exception as e:
        print(f"  ⚠️ {key.upper()} API 실패 ({e}) → CSV fallback")
        try:
            macro_raw[key] = load_fallback_csv(key, meta_dir)
            sources[key] = 'CSV'
        except Exception:
            print(f"  ⚠️ {key.upper()} CSV 없음 → 빈 Series 사용")
            macro_raw[key] = pd.Series(dtype=float)
            sources[key] = 'EMPTY'

# KOSPI 거래일 기준 데이터 정렬 (ffill 기반 보간)
aligned = {'kospi': kospi_series}
for key in ['sp500', 'usd_krw', 'vix']:
    s = macro_raw[key].copy()
    s.index = pd.to_datetime(s.index)
    if s.empty:
        aligned[key] = pd.Series(np.nan, index=kospi_index, name=key)
        continue
    combined_index = s.index.union(kospi_index).sort_values()
    aligned[key] = s.reindex(combined_index).ffill().reindex(kospi_index)

df_macro = pd.DataFrame(aligned)
df_macro.index.name = 'Date'

print(f"\n✅ 수집 및 KOSPI 정렬 완료: {len(df_macro):,} 거래일")

In [ ]:
# ==========================================
# 2. Market Regime 및 파생 피처 계산
# ==========================================
print("⚙️ Market Regime 및 캘린더 피처 계산 중...")

# 1. KOSPI 기술적 지표 계산
df_macro['kospi_ma200'] = df_macro['kospi'].rolling(window=200).mean()
df_macro['kospi_vol20'] = df_macro['kospi'].pct_change().rolling(window=20).std()

# 동적 임계값: 최근 1년(250거래일) 변동성 중위수
vol_median = df_macro['kospi_vol20'].rolling(window=250).median()

# 2. Regime 판단 로직
#   - Bull (1) : 주가가 200일선 위
#   - Bear (-1): 주가가 200일선 아래 & 단기 변동성이 장기 중위수보다 큼 (투매 장세)
#   - Neutral (0): 그 외 횡보장
conditions = [
    (df_macro['kospi'] > df_macro['kospi_ma200']),
    (df_macro['kospi'] < df_macro['kospi_ma200']) & (df_macro['kospi_vol20'] > vol_median)
]
choices = [1, -1]
df_macro['market_regime'] = np.select(conditions, choices, default=0)

# 3. 미국 시장 수익률
# 한국 시초가에 영향을 주는 전일(혹은 직전 거래일) 미국 시장 수익률
df_macro['us_return_1d'] = df_macro['sp500'].pct_change().shift(1)

# 4. 결측치 제거 (초기 250일분 이동평균 계산 구간 제거)
df_macro = df_macro.dropna()

print("✅ 계산 완료. Regime 분포:")
print(df_macro['market_regime'].value_counts(normalize=True).map('{:.1%}'.format))

In [ ]:
# ==========================================
# 3. 데이터 저장
# ==========================================
# 인덱스(Date)를 컬럼으로 리셋하여 병합하기 편하게 만듭니다.
df_macro_final = df_macro.reset_index()
if 'Date' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'Date': 'date'})
elif 'index' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'index': 'date'})

# 필요한 핵심 피처만 선택 (KOSPI MA 등은 레짐 계산용이므로 제외 가능하나 분석용으로 남겨둠)
final_cols = ['date', 'kospi', 'usd_krw', 'vix', 'us_return_1d', 'market_regime']
df_save = df_macro_final[final_cols].copy()

# Parquet 포맷으로 저장
meta_dir.mkdir(parents=True, exist_ok=True)
df_save.to_parquet(macro_filepath, index=False)
# CSV 포맷으로 저장
csv_filepath = meta_dir / "macro_regime.csv"
df_save.to_csv(csv_filepath, index=False)

print(f"💾 매크로 데이터 저장 완료! -> {macro_filepath}")
print(f"   - 총 데이터 수: {len(df_save):,}일")
display(df_save.head())